# Hands-on · one HCC core, three modalities · from the raw data
**MERIT (CNRS) workshop · spatial multimodal integration**

Same FFPE section, three instruments: **H&E**, **Xenium Prime 5K** (5,001 genes, 8,290 cells) and **COMET** (DAPI + 15 proteins).
Two packages, two levels:

* **multispace** (cell level): one SpatialData object with a fixed data contract, image registration, cell typing, spatial statistics.
  multispace is our in-house package (not yet published). It grew out of **MOSAIK** (Baptista et al., github.com/anthbapt/MOSAIK)
  and contains MOSAIK's resegmentation code.
* **spatiomic** (pixel level): self-organising map + Leiden clustering of the COMET pixels, with no segmentation.

| # | Task |
|---|---|
| 1 | Load the object, look at the three layers |
| 2 | Registration and segmentation QC (multispace) |
| 3 | Pixel clusters on a COMET crop (spatiomic) |
| 4 | Pixels → cells, then a joint embedding vs RNA only |
| 4b | One multilayer network on the same cells (optional) |
| 5 | In pairs: RNA vs protein  |
| 5b | Cell typing with RNA + protein markers (optional) |
| 5c | Who sits next to whom: neighbourhood enrichment |
| 6 | Tumour vs non-tumour: core01 vs core02 (precomputed) |

**This version needs nothing precomputed.** Part 0 rebuilds everything from the raw data in `data/` (registration of the
full H&E, cell typing, spatiomic on both whole cores, the joint spatiomic model, RNA + protein cell typing, figures) and
writes it to `results_from_raw/`. Then sections 1–6 are the session notebook, reading from there.
Part 0 takes **about 20 min and ~6 GB of RAM** the first time; each step is skipped when its output already exists, so
re-running the notebook takes about 2 min. Resegmentation is not run in this
session (no Xenium morphology image ships with this dataset, and it needs a GPU): section 2 shows results obtained beforehand. Everything in this notebook runs on a laptop CPU in a few seconds per cell.

**Run it from the `workshop/` folder**, next to `data/` (raw data: `core01_hcc.zarr`, `core02_non_tumour.zarr`) and
`handson/` (`workshop_hcc.py` and the preparation steps), in the `merit` environment.

In [ ]:
# --- CONFIG -------------------------------------------------------------------
import sys
from pathlib import Path
sys.path.insert(0, "handson")      # workshop_hcc.py, prepare_core01.py, prepare_core02.py, figure scripts
DATA = Path("data")                # raw data: core01_hcc.zarr, core02_non_tumour.zarr (Zenodo 21713660)
PREP = Path("results_from_raw")    # everything this notebook computes (~1.5 GB)
FIG = PREP / "figures"
ZARR = PREP / "core01_ms.zarr"
# A 1000 × 1000 px window (COMET pixels, ~280 µm): tumour nests, stroma and a lymphoid aggregate
CROP = dict(y0=2000, y1=3000, x0=1800, x1=2800)
RUN_REGISTRATION = False           # the whole-H&E registration runs in part 0; True re-runs it on the object (~1-3 min)

In [ ]:
import warnings, json; warnings.filterwarnings("ignore")
from IPython.display import Image, display
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import scanpy as sc, spatialdata as sd, spatialdata_plot  # noqa: F401 (adds .pl)
import multispace as ms
import spatiomic as so
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from workshop_hcc import *   # helpers for this hands-on (pixel clustering, pixels → cells, metrics)
sc.settings.verbosity = 0

## 0 · Rebuild everything from the raw data (~20 min the first time)
The session notebook loads these results from a downloaded `prepared/` folder. Here they are computed, with the same
functions (`handson/prepare_core01.py`, `prepare_core02.py`; open them to see each step in full). Each step writes its
output to `results_from_raw/` and is skipped next time if the output is there.

**0.1 · core01: data contract, H&E registration, cell typing.** The raw object is renamed to the multispace contract
and written to `results_from_raw/core01_ms.zarr`. The H&E is registered onto the COMET DAPI with `ms.register_image`;
each try is scored (H&E haematoxylin vs DAPI correlation) and the best is kept. Cells are typed with the marker strategy
(`annotate_by_markers`: Leiden on RNA at resolution 2.0, signed markers, one label per cluster). ~3 min.

In [ ]:
import prepare_core01 as p1
PREP.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
s1 = p1.step_contract(DATA / "core01_hcc.zarr", PREP, force=False, core="core01_hcc")

**0.2 · core01: spatiomic on the whole core.** COMET at half resolution, tissue mask, SOM 25 × 25 on 60,000 pixels,
Leiden on the SOM nodes, every pixel labelled; cluster profiles and the vicinity graph. ~3 min.

In [ ]:
lab1, mask1, arr1, ch, img_n1 = p1.step_pixels(s1, PREP, force=False)
del arr1, img_n1

**0.3 · core01: pixels → cells, joint embedding, metrics, spatial analysis, RNA + protein cell typing.**
Each Xenium cell gets the fraction of its pixels in each spatiomic cluster; RNA, protein and pixel blocks are scaled and
compared (`integration_metrics.csv`); multispace's spatial analysis runs on the cell types; `annotate_multiomic` adds the
protein markers, and the Xenium–COMET segmentation match is computed. ~2 min.

In [ ]:
p1.step_integration(s1, lab1, PREP, force=False)
p1.step_analysis(s1, PREP)
p1.step_multiomic(s1, PREP, force=False)
del s1, lab1, mask1

**0.4 · core02 (non-tumour) and the joint spatiomic model.** The same steps on core02, then the comparison: core02
labelled with the core01-only model (the diagnostic), and one spatiomic model fitted on both cores, with a shared
normalisation and with per-core normalisation. ~8 min.

In [ ]:
import gc
import prepare_core02 as p2
C2 = PREP / "core02"; C2.mkdir(exist_ok=True)
s2 = p2.step_contract(DATA / "core02_non_tumour.zarr", C2, force=False, core="core02_non_tumour")
if not (C2 / "joint" / "run.json").exists():
    ref1, _ = p2.refit_core01_model(DATA / "core01_hcc.zarr", PREP)
    lab2 = p2.step_pixels(sd.read_zarr(ZARR), s2, ref1, C2)
    del ref1; gc.collect()
else:
    lab2 = np.load(C2 / "joint" / "core02_labelled_ds2.npy")
p2.step_integration(s2, lab2, C2, force=False)
p2.step_analysis(s2, C2)
del s2, lab2; gc.collect()

**0.5 · Figures used below** (the normalisation pitfall, and RNA-only vs RNA + protein cell types).

In [ ]:
from make_figures import comparison_figures
import make_celltype_figures
comparison_figures(PREP, FIG)
make_celltype_figures.main(PREP, FIG)
%matplotlib inline

## 1 · Load the object and look at the three layers

In [ ]:
sdata = sd.read_zarr(ZARR)
sdata

Everything sits in one coordinate system, `global` = the COMET pixel grid (1 px ≈ 0.28 µm).
The Xenium elements and the H&E reach it through an affine transform. The table follows the **multispace contract**:
RNA counts in `X`, the 16 COMET intensities in `obsm['protein']`, cell types in `obs['cell_type']`.

**Cell types** come from the multispace *marker strategy* (`annotate_clusters_by_markers`): Leiden clusters on the RNA
(resolution 2.0, 23 clusters), a signed score per cluster for each cell type (positive minus negative markers,
`LIVER_MARKERS_SIGNED` in `workshop_hcc.py`), and one label per cluster. `cell_type_confidence` is the cluster's
winner / runner-up score ratio.

> **Caveat on the markers.** The Xenium Prime 5K panel lacks several textbook liver markers: **ALB, KRT19, COL1A1, PTPRC**
> (and GPC3, AFP, KRT7, APOA1, CYP3A4). We use substitutes: APOB, AHSG, CPS1, ARG1 and HNF4A for hepatocytes; EPCAM, SOX9 and
> CFTR for cholangiocytes; PDGFRB, RGS5 and COL4A1 for stroma; immune lineage genes instead of PTPRC. **HCC tumour vs normal
> hepatocyte** rests on oncofetal genes (IGF2, DLK1, PEG10, H19) against CPS1/ARG1, and the separation is weak (winner /
> runner-up ratio ≈ 1.1 for most hepatocyte-like clusters). Treat "HCC tumour" and "Hepatocyte" as one hepatocyte-lineage
> compartment unless the protein layer (HepPar1, CK8) or the H&E supports the split. Good enough for this workshop, not for a paper.
> Section 5b adds the protein markers to the same strategy.

In [ ]:
from spatialdata.transformations import get_transformation
for name in ["if", "he", "cell_boundaries", "transcripts"]:
    el = sdata[name]
    print(f"{name:16s}", {cs: type(t).__name__ for cs, t in get_transformation(el, get_all=True).items()})
t = sdata.tables["table"]
print(t)
pd.concat([t.obs["cell_type"].value_counts().rename("cells"),
           t.obs.groupby("cell_type", observed=True)[["CD20_-_TRITC_positive", "CD45_-_TRITC_positive", "CK8_-_TRITC_positive"]]
            .mean().round(2).add_prefix("protein+ ")], axis=1)   # does the protein agree with the RNA-based labels?

In [ ]:
crop = sd.bounding_box_query(sdata, axes=("x", "y"), min_coordinate=[CROP["x0"], CROP["y0"]],
                             max_coordinate=[CROP["x1"], CROP["y1"]], target_coordinate_system="global")
fig, axs = plt.subplots(1, 2, figsize=(13, 6.5))
crop.pl.render_images("he").pl.show(coordinate_systems="global", ax=axs[0], title="H&E (registered)")
(crop.pl.render_images("if", channel=["CK8 - TRITC", "CD45 - TRITC", "αSMA - TRITC"], palette=["red", "cyan", "lime"])
     .pl.render_shapes("cell_boundaries", fill_alpha=0, outline_alpha=0.7, outline_color="white", outline_width=0.6)
     .pl.show(coordinate_systems="global", ax=axs[1], title="COMET CK8 · CD45 · αSMA + Xenium cells"))

**Question.** Do the Xenium cell outlines sit on the COMET nuclei? Where would you expect segmentation to struggle?

## 2 · Registration and segmentation QC with multispace 
The H&E was registered onto the COMET DAPI with `ms.register_image(sdata, "he", "morphology_focus", method="affine")`
(intensity-based, normalised mutual information). The dataset also ships a keypoint-based transform, kept as the coordinate
system `he_reference`: how far apart are the two?

In [ ]:
if RUN_REGISTRATION:
    res = ms.register_image(sdata, "he", "morphology_focus", method="affine", max_side=1000)
    print(res.details.get("quality"))
reg = sdata.attrs["multispace"]["registration"]["he"]
print("method:", reg["method"], "| fixed:", reg["fixed"])
registration_error(sdata)   # displacement between the two transforms over the whole H&E, in COMET px and µm

In [ ]:
from shapely.affinity import affine_transform

def in_global(key):
    g = sdata.shapes[key]
    M = get_transformation(g, to_coordinate_system="global").to_affine_matrix(("x", "y"), ("x", "y"))
    return g.geometry.apply(lambda p: affine_transform(p, [M[0, 0], M[0, 1], M[1, 0], M[1, 1], M[0, 2], M[1, 2]]))

um2 = COMET_UM_PER_PX ** 2
seg = {name: in_global(key) for name, key in [("Xenium default", "cell_boundaries"), ("COMET InstaSeg", "comet_cell_masks")]}
pd.DataFrame({name: {"cells": len(g), "median area (µm²)": round(float(np.median(g.area)) * um2, 1)} for name, g in seg.items()}).T

**Question.** Same tissue, 8,290 vs 6,418 cells. Which segmentation would you trust for protein quantification, and why?

### Resegmentation: results shown, not run
multispace (from MOSAIK) resegments Xenium data in two steps: **Cellpose-SAM** on the morphology image gives a prior, then **Proseg**
moves the cell boundaries to fit the transcripts. This HCC dataset ships without the Xenium morphology image, and Proseg needs
~10 min per core on a GPU node, so here are the results on a breast-cancer Xenium core (METABRIC, MB-GU-1017) run on the HPC.

* Proseg keeps cells centred on their Cellpose prior (median shift 0.02–0.08 px) and assigns ~80% of transcripts.
* The validation caught a coordinate bug in the earlier code: transcripts were shifted by up to 34 px (7 µm), so in one cohort
  66% of transcripts had been counted in the wrong cell. Every run now checks its own registration (shift scan peaks at 0, 0).

In [ ]:
# Not rebuilt here: resegmentation needs the Xenium morphology image (not in this dataset) and a GPU.
RESEG = Path("handson/prepared/reseg_metabric")   # results of the METABRIC validation run
display(Image(RESEG / "pilot_MB-GU-1017_5_proseg.png", width=1000))     # Cellpose prior vs Proseg cells, QC
display(Image(RESEG / "pilot_MB-GU-1017_6_old_vs_new.png", width=1000))  # old (buggy) vs corrected run

**Question.** Proseg assigns fewer transcripts than Cellpose (79% vs 92%). Is that a problem? What happens to the rest?

## 3 · Pixel clusters on COMET with spatiomic 
No cells here: every pixel is a 15-marker vector (DAPI left out). Steps from the spatiomic *full example*:
subsample → clip → normalise → self-organising map → kNN graph + Leiden on the SOM nodes → label every pixel.

In [ ]:
arr, channels = comet_array(sdata, downscale=2, crop=CROP)      # (y, x, 15), crop downscaled ×2
# tissue mask from the whole core (a threshold computed on the crop alone would drop the dim stroma)
mask = np.load(PREP / "tissue_mask_ds2.npy")[CROP["y0"] // 2:CROP["y1"] // 2, CROP["x0"] // 2:CROP["x1"] // 2]
res = run_spatiomic(arr, mask, n_pixels=20_000, som_size=(15, 15), iterations=10, resolution=1.0)
labelled = res["labelled"]
print(f"{labelled.max() + 1} clusters | quantisation error {res['quantization_error']:.3f} | modularity {res['modularity']:.2f}")

prof = cluster_profiles(res["img_n"], labelled, channels)
fig, axs = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [1, 1.6]})
axs[0].imshow(np.ma.masked_less(labelled, 0), cmap="tab10", interpolation="nearest"); axs[0].set_axis_off()
axs[0].set_title("spatiomic pixel clusters")
P = prof.drop(columns="n_pixels"); P.columns = [c.split(" - ")[0] for c in P.columns]
im = axs[1].imshow(((P - P.mean()) / P.std()).values, cmap="RdBu_r", vmin=-2.5, vmax=2.5, aspect="auto")
axs[1].set_xticks(range(P.shape[1]), P.columns, rotation=60, ha="right"); axs[1].set_yticks(range(len(P)), P.index)
axs[1].set_title("cluster × marker (z-score)"); plt.colorbar(im, ax=axs[1]); plt.tight_layout()

In [ ]:
vc, graph, _ = vicinity(labelled, order=2)
(vc.div(vc.sum(axis=1), axis=0) * 100).round(0)   # row = cluster, column = % of its neighbouring pixels in each cluster

**Question.** Name the clusters from their marker profiles (tumour epithelium, stroma, lymphoid, vessel…).
Which clusters touch which? Is there a cluster that looks bright in *every* channel, and what could it be?

*On the whole core (part 0, 25 × 25 SOM): `results_from_raw/spatiomic_cluster_profiles.csv` and `results_from_raw/spatiomic_vicinity.csv`.*
The crop above has its own small SOM, so its cluster numbers differ from the whole-core ones. Check your names against ours below.

In [ ]:
full = pd.read_csv(PREP / "spatiomic_cluster_profiles.csv", index_col=0)
full = full.drop(columns="n_pixels")
full.columns = [c.split(" - ")[0] for c in full.columns]
full.index = [pixel_cluster_label(k) for k in full.index]   # names: PIXEL_CLUSTER_NAMES in workshop_hcc.py
((full - full.mean()) / full.std()).round(1).style.background_gradient(cmap="RdBu_r", vmin=-2.5, vmax=2.5, axis=None)

## 4 · From pixels to cells, then a joint embedding
The bridge between the two levels: for each Xenium cell, the fraction of its pixels in each spatiomic cluster.
That gives a third cell-level block, next to RNA and protein.

In [ ]:
geo = in_global("cell_boundaries")
inside = geo.centroid.x.between(CROP["x0"], CROP["x1"]) & geo.centroid.y.between(CROP["y0"], CROP["y1"])
ids = list(geo.index[inside].astype(str))
cell_mask = rasterize_cells(geo[inside].values, ids, labelled.shape, origin=(CROP["x0"], CROP["y0"]), scale=2)
comp = pixel_composition(cell_mask, labelled, ids)

a = t[t.obs["cell_id"].astype(str).isin(comp.index)].copy()
comp = comp.reindex(a.obs["cell_id"].astype(str)).fillna(0.0)
a.X = a.layers["counts"].copy(); sc.pp.normalize_total(a); sc.pp.log1p(a); sc.pp.pca(a, n_comps=20)
prot = np.log1p(a.obsm["protein"].drop(columns="DAPI")); prot = (prot - prot.mean()) / prot.std()

# Early fusion with block scaling: each modality contributes the same total variance
blocks = [block_scale(a.obsm["X_pca"]), block_scale(prot.values), block_scale(comp.values)]
a.obsm["X_joint"] = PCA(20, random_state=0).fit_transform(np.hstack(blocks))
for rep in ["X_pca", "X_joint"]:
    sc.pp.neighbors(a, use_rep=rep, key_added=rep); sc.tl.leiden(a, neighbors_key=rep, key_added=f"leiden_{rep}",
                                                             resolution=0.8, flavor="igraph", n_iterations=2)
dominant = comp.values.argmax(1); xy = a.obsm["spatial"]
summary = pd.DataFrame({name: {"clusters": a.obs[col].nunique(),
                               "spatial coherence": spatial_coherence(xy, a.obs[col].astype(str).values),
                               "ARI vs pixel clusters": adjusted_rand_score(a.obs[col], dominant)}
                        for name, col in [("RNA only", "leiden_X_pca"), ("joint", "leiden_X_joint")]}).T
print(f"{a.n_obs} cells in the crop"); summary.round(2)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 5.2))
for ax, (vals, title) in zip(axs, [(a.obs["leiden_X_pca"], "RNA only"), (a.obs["leiden_X_joint"], "joint (RNA + protein + pixels)"),
                                   (dominant, "dominant spatiomic cluster")]):
    ax.scatter(xy[:, 0], xy[:, 1], c=pd.Categorical(vals).codes, cmap="tab20", s=14)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_axis_off(); ax.set_title(title)
pd.read_csv(PREP / "integration_metrics.csv").round(2)   # the same, on the whole core (with a shuffled-layer control)

**Question.** The shuffled control (protein and pixel blocks permuted across cells) loses most of its spatial coherence:
what does that tell you? Why might the RNA-only clustering look *more* spatially coherent than the joint one?

## 4b · One multilayer network on the same cells (optional)
Another way to fuse: each modality becomes a graph **layer** on the same nodes (the 8,290 Xenium cells): cells are linked
when they are similar in RNA, in protein, or in pixel composition. The shared segmentation is what makes the nodes the same.
Two questions: do the layers agree on who is similar, and what do they find together?

In [ ]:
t = sdata.tables["table"]
rna = t.copy(); rna.X = rna.layers["counts"].copy()
sc.pp.normalize_total(rna); sc.pp.log1p(rna); sc.pp.pca(rna, n_comps=30, random_state=0)
prot = t.obsm["protein"].drop(columns=["DAPI"]); prot = prot.fillna(prot.median())
prot_z = ((np.log1p(prot) - np.log1p(prot).mean()) / np.log1p(prot).std()).values
comp_all = pd.read_csv(PREP / "pixel_composition.csv", index_col=0).reindex(t.obs["cell_id"].astype(str)).fillna(0).values
layers = {"RNA": knn_layer(rna.obsm["X_pca"]), "protein": knn_layer(prot_z), "pixels": knn_layer(comp_all)}
layer_agreement(layers).round(3)   # Jaccard overlap of the kNN edges: how often two modalities agree on a neighbour

In [ ]:
xy, ct = t.obsm["spatial"], t.obs["cell_type"].astype(str).values
views = {name: multiplex_leiden({name: A}) for name, A in layers.items()}
views["multilayer"] = multiplex_leiden(layers)          # communities shared by the three layers
pd.DataFrame({k: {"communities": len(set(v)), "spatial coherence": round(spatial_coherence(xy, v.astype(str), 10), 3),
                  "ARI vs cell types": round(adjusted_rand_score(ct, v), 3)} for k, v in views.items()}).T

**Network exploration.** A random walk with restart (MultiXrank, Baptista et al., *Commun Phys* 2022) starts from seed
cells, moves along edges within a layer, jumps between layers, and keeps returning to the seeds. Cells that score high are
close to the seeds *in all modalities at once*. Seeds here: the B cells. No cell coordinates are used.

In [ ]:
seeds = np.where(ct == "B cell")[0]
score = multiplex_rwr(layers, seeds, restart=0.7, delta=0.5)
top = [i for i in np.argsort(-score) if i not in set(seeds)][:300]
print("top-300 neighbours of the B cells:", pd.Series(ct[top]).value_counts().head(4).to_dict())
from scipy.spatial import cKDTree
d_top = cKDTree(xy[seeds]).query(xy[top])[0] * COMET_UM_PER_PX
d_rnd = cKDTree(xy[seeds]).query(xy[np.random.default_rng(0).choice(len(ct), 300, replace=False)])[0] * COMET_UM_PER_PX
print(f"median distance to the nearest B cell: top-300 {np.median(d_top):.0f} µm, random cells {np.median(d_rnd):.0f} µm")
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(xy[:, 0], xy[:, 1], c="#DDDDDD", s=2)
ax.scatter(xy[top, 0], xy[top, 1], c="#2457A6", s=6, label="top 300 (random walk)")
ax.scatter(xy[seeds, 0], xy[seeds, 1], c="#B4541F", s=6, label="seeds: B cells")
ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_axis_off(); ax.legend(loc="lower left")

**Question.** The layers share only a few % of their edges, yet the walk lands next to the B cells, in tissue space,
without ever seeing coordinates. What does that say about the lymphoid aggregate? Try other seeds (macrophages,
endothelial cells), or drop one layer from `layers` and compare.

## 5 · In pairs: RNA vs protein

In [ ]:
counts = pd.DataFrame(t.layers["counts"], index=t.obs_names, columns=t.var_names)
conc = rna_protein_concordance(counts, t.obsm["protein"])
ax = conc.plot.barh(x="gene", y="spearman", legend=False, figsize=(6, 4), color="#2457A6"); ax.invert_yaxis()
ax.set_xlabel("Spearman across all 8,290 cells"); conc.round(2)

In [ ]:
# Sparse RNA makes per-cell correlations low. Another view: among protein-positive cells, how often is the RNA detected?
pos = {"MS4A1": "CD20_-_TRITC_positive", "VEGFA": "VEGF_-_TRITC_positive", "CDH1": "E-Cadherin_-_TRITC_positive"}
pd.DataFrame({g: {"RNA detected | protein+": (counts[g][t.obs[c]] > 0).mean(),
                  "RNA detected | protein−": (counts[g][~t.obs[c]] > 0).mean()} for g, c in pos.items()}).T.round(2)

Discuss with your neighbour:

* Which markers agree between RNA and protein, and which don't? Is the disagreement **biology** (secretion, post-transcriptional
  regulation, half-life) or **technology** (segmentation, spillover from neighbours, probe or antibody quality, RNA dropout)?
* Do pixel clusters split cells that the RNA clustering lumps together? Find one such cell and look at it in the images (section 1).
* What does the protein layer show that the transcriptome alone doesn't?

## 5b · Cell typing with RNA + protein markers (optional)
The labels above use RNA only. `annotate_multiomic` (in `workshop_hcc.py`) keeps the multispace marker logic and adds the
protein layer:

1. **Clusters on RNA + protein**: 30 RNA PCs and the 15 COMET intensities (log, z-scored per channel), each block scaled to
   the same total variance; Leiden at resolution 2.0, as for the RNA-only labels.
2. **A protein score next to the RNA score.** Protein decides the *compartment*, where the 5K panel is weakest:
   immune = CD45 (instead of the missing PTPRC), epithelial = CK8, E-cadherin, HepPar1, stromal = vimentin. Each score is a
   contrast (own compartment minus the others), after removing each cell's overall brightness. Within a compartment, one
   specific protein where the panel has one: CD20/CD45RA (B), CD206/FXIIIa (macrophage), CD1c (DC), αSMA (fibroblast).
3. **Score = ½ RNA + ½ protein**, then one label per cluster with multispace's quality gates.

Configuration: `PROTEIN_LINEAGES`, `PROTEIN_SPECIFIC`, `CELL_TYPE_LINEAGE` in `workshop_hcc.py`.

In [ ]:
mo = annotate_multiomic(t, protein_weight=0.5)          # writes obs['cell_type_multiomic']
print(f"{(t.obs['cell_type'].astype(str) == t.obs['cell_type_multiomic'].astype(str)).mean():.0%} of cells keep their RNA label")
pd.crosstab(t.obs["cell_type"], t.obs["cell_type_multiomic"], rownames=["RNA only"], colnames=["RNA + protein"])

In [ ]:
# Each modality's own winner per cluster: where RNA and protein disagree, which one won?
mo[["n_cells", "rna_winner", "protein_winner", "cell_type_multiomic", "ambiguity_ratio"]].sort_values("cell_type_multiomic")

In [ ]:
# From part 0, with the Xenium–COMET segmentation match: are label changes more frequent where the segmentations disagree?
cm = pd.read_csv(PREP / "cell_typing_multiomic" / "cells.csv", index_col=0)
print(pd.crosstab(cm["change"], cm["one_to_one"], normalize="columns").round(3))
display(Image(str(FIG / "18_multiomic_cell_types.png"), width=1000))

**Questions.**

* 165 cells go from "HCC tumour" (RNA) to an immune type (protein: CD45+, CK8−). Find a few
  (`t.obs.query("cell_type == 'HCC tumour' and cell_type_multiomic != 'HCC tumour'")`) and look at them in the images
  (section 1): immune cells inside the tumour, or tumour cells picking up a neighbour's CD45?
* Compartment changes happen in 7% of cells without a one-to-one COMET cell vs 3% of matched cells. What does that tell you
  about where the protein value of a cell comes from?
* Re-run with `protein_weight=0.25` and `0.75`. Which labels are stable?
* Run `PROTEIN_SPECIFIC["Neutrophil"] = ["CD66"]`, then `annotate_multiomic(t)` again. What happens to the lymphoid
  aggregate, and why? (Hint: the CD66 antibody also sees CEACAM1, found on activated T cells.) `del PROTEIN_SPECIFIC["Neutrophil"]` to undo.

## 5c · Who sits next to whom: neighbourhood enrichment
Cell types answer *what* is there; this asks *who sits next to whom* (slide 5: colocalisation). multispace's
`run_spatial_analysis` builds a Delaunay graph on the cell centroids (each cell linked to its touching neighbours), counts
the links between every pair of cell types, and compares them with 1,000 random shuffles of the labels (squidpy
`nhood_enrichment`). **z > 0**: the two types touch more often than chance; **z < 0**: they avoid each other.
Everything below runs live, in a few seconds.

In [ ]:
# core01, RNA-only labels, with multispace (the same call works on any SpatialData object in the multispace contract)
res = ms.run_spatial_analysis(sdata, results_dir=PREP / "nhood_rna", sections=["neighborhood"], cluster_key="cell_type")
z_rna = res["nhood_enrichment_zscore"]

# core01, RNA + protein labels (section 5b). If you skipped 5b, the labels come from part 0.
if "cell_type_multiomic" not in t.obs:
    t.obs["cell_type_multiomic"] = pd.read_csv(PREP / "cell_typing_multiomic" / "cells.csv", index_col=0) \
        .loc[t.obs_names, "cell_type_multiomic"].to_numpy()
z_mo = ms.run_spatial_analysis(sdata, results_dir=PREP / "nhood_multiomic", sections=["neighborhood"],
                               cluster_key="cell_type_multiomic")["nhood_enrichment_zscore"]

# core02 (non-tumour): no SpatialData object here, only its cell table (centroids + RNA labels): same settings
obs2 = pd.read_csv(PREP / "core02" / "integration_obs.csv", index_col=0)
z_02 = nhood_zscore(obs2[["x", "y"]].to_numpy(), obs2["cell_type"])

In [ ]:
order = ["HCC tumour", "Hepatocyte", "Cholangiocyte", "Fibroblast / stellate", "Endothelial", "Macrophage",
         "T cell", "B cell", "Dendritic cell"]
fig, axs = plt.subplots(1, 3, figsize=(19, 5.8), gridspec_kw={"wspace": 0.6})
for ax, (z, ttl) in zip(axs, [(z_rna, "core01 HCC · RNA labels"), (z_mo, "core01 HCC · RNA + protein labels"),
                              (z_02, "core02 non-tumour · RNA labels")]):
    im = plot_nhood(z, ax, ttl, order=order)
fig.colorbar(im, ax=axs, shrink=0.7, label="neighbourhood z-score"); plt.show()

In [ ]:
# A few pairs side by side (NaN: one of the two types is absent from that core)
pairs = [("T cell", "B cell"), ("T cell", "HCC tumour"), ("Macrophage", "HCC tumour"), ("Macrophage", "Hepatocyte"),
         ("Macrophage", "Fibroblast / stellate"), ("Endothelial", "Fibroblast / stellate")]
get = lambda z, a, b: z.loc[a, b] if a in z.index and b in z.index else np.nan
pd.DataFrame({f"{a} – {b}": {"core01 RNA": get(z_rna, a, b), "core01 RNA + protein": get(z_mo, a, b),
                              "core02 RNA": get(z_02, a, b)} for a, b in pairs}).T.round(1)

**Questions.**

* On the HCC core, T cells avoid tumour cells (z ≈ −86) and sit with B cells (z ≈ +38). Immune exclusion, or simply
  two compartments (tumour nests vs the lymphoid aggregate)? What would you need to tell them apart?
* Macrophages avoid tumour nests in core01 but mix with hepatocytes in core02 (Kupffer cells live in the sinusoids).
  Same label, different neighbourhood: what does that say about using one "macrophage" class across conditions?
* Do the RNA + protein labels change any conclusion? Which pairs move most, and is it the cell typing or the biology?
* z-scores grow with the number of cells: compare **signs and ranks** between cores, not the raw values. And it is one
  core per condition: an illustration, not a test.

## 6 · Tumour vs non-tumour: core01 vs core02 
`core02_non_tumour` is non-tumour liver from the same TMA slide, prepared with `prepare_core02.py` (same steps as core01).
To compare pixel clusters, **one** spatiomic model was fitted on pixels pooled from both cores (30,000 per core), so
cluster *k* is the same phenotype in both. Two things went wrong on the way, and they are worth seeing:

* **The core01 model does not describe core02.** Labelling core02 with the core01-only model gives a quantisation error of
  0.45, against 0.13 on core01: normal liver contains pixel phenotypes the tumour core never showed the model.
* **core02 is 2–5× brighter in every channel.** With one shared normalisation, 8 of the 11 joint clusters come ≥ 90% from a
  single core: the model separates the cores by brightness (autofluorescence, staining, exposure), not by biology. Normalising
  each core with its own percentiles first fixes this, and removes any true global difference along with it.

In [ ]:
C2 = PREP / "core02"
print(json.load(open(C2 / "joint" / "run.json")))   # quantisation errors of the three models
display(Image(str(FIG / "15_normalisation_pitfall.png"), width=900))

In [ ]:
obs1 = pd.read_csv(PREP / "integration_obs.csv", index_col=0)
obs2 = pd.read_csv(C2 / "integration_obs.csv", index_col=0)
cells = pd.concat({"core01 HCC": obs1["cell_type"].value_counts(normalize=True),
                   "core02 non-tumour": obs2["cell_type"].value_counts(normalize=True)}, axis=1).fillna(0)
def pix_share(path):
    lab = np.load(path); lab = lab[lab >= 0]
    return pd.Series(np.bincount(lab, minlength=len(JOINT_CLUSTER_NAMES)) / lab.size,
                     index=[f"{k} · {v}" for k, v in JOINT_CLUSTER_NAMES.items()])
pixels = pd.concat({"core01 HCC": pix_share(C2 / "joint" / "core01_labelled_ds2.npy"),
                    "core02 non-tumour": pix_share(C2 / "joint" / "core02_labelled_ds2.npy")}, axis=1)
fig, axs = plt.subplots(1, 2, figsize=(15, 5.5))
(cells * 100).sort_values("core01 HCC").plot.barh(ax=axs[0], color=["#B4541F", "#2457A6"]); axs[0].set_xlabel("% of cells")
axs[0].set_title("Cell types (RNA, marker strategy)")
(pixels * 100).iloc[::-1].plot.barh(ax=axs[1], color=["#B4541F", "#2457A6"]); axs[1].set_xlabel("% of tissue pixels")
axs[1].set_title("Pixel clusters (COMET, joint model, per-core normalisation)"); plt.tight_layout()

**Question.** Which compartments change between tumour and non-tumour liver? Do the RNA-based cell types and the
segmentation-free pixel clusters tell the same story? Where do they disagree, and which would you believe?
One core per condition: this is an illustration, not a test.